In [2]:
import pandas as pd
import os
import json
import glob

In [1]:
class match_summary:
    def __init__(self, *json_folder):
        self.json_folder = json_folder
        self.match_list = self.get_files()
        


    def get_files(self):
        all_files = []
        for root, dris, files in os.walk(self.json_folder):
            files = glob.glob(os.path.join(root,'*.json'))
            for f in files:
                all_files.append(os.path.abspath(f))
        return all_files
    
    def load_json(self):
        tournament_list = []
        for match_detail in self.match_list:
            with open (match_detail, 'r', encoding='utf-8') as doc:
                exp = json.load(doc)
                tournament_list.append(exp)
        return tournament_list

    def get_over_data(self):
        tournament_list = self.load_json()
        match_dataframes = []

        for exp in tournament_list:
            match_info = exp.get('info', {})
            match_details = {
                "city": match_info.get("city", "Unknown"),
                "date": match_info.get("dates", ["Unknown"])[0],
                "event_name": match_info.get("event", {}).get("name", "Unknown"),
                "match_type": match_info.get("match_type", "Unknown"),
                "winner": match_info.get("outcome", {}).get("winner", "Unknown"),
                "player_of_match": ", ".join(match_info.get("player_of_match", ["Unknown"]))
            }


            innings = exp.get('innings',[])
            deliveries_list = []
            
            for inning_data in innings:
                inning_name = inning_data.get('team', 'Unknown Team')
                overs = inning_data.get('overs', [])
                
                for over_data in overs:
                    over_number = over_data.get('over', None)
                    deliveries = over_data.get('deliveries', [])
                    
                    for delivery in deliveries:
                        delivery['inning'] = inning_name
                        delivery['over'] = over_number
                        
                        runs = delivery.get('runs', {})
                        delivery['runs_batter'] = runs.get('batter', 0)
                        delivery['runs_extras'] = runs.get('extras', 0)
                        delivery['runs_total'] = runs.get('total', 0)
                        
                        full_delivery = {**match_details, **delivery}
                        deliveries_list.append(full_delivery)

            df = pd.DataFrame(deliveries_list)
            
            col_to_remove = ['extras', 'wickets', 'replacements', 'review', 'runs']
            df = df.drop(columns=[col for col in col_to_remove if col in df.columns], errors='ignore')
            match_dataframes.append(df)

        final_df = pd.concat(match_dataframes, ignore_index=True)

        return final_df 

In [5]:
json_folder = 'ipl_json'
match_sum = match_summary(json_folder)
ipl_data_table = match_sum.get_over_data()

In [6]:
ipl_data_table

,city,date,event_name,match_type,winner,player_of_match,batter,bowler,non_striker,inning,over,runs_batter,runs_extras,runs_total
0,Hyderabad,2017-04-05,Indian Premier League,T20,Sunrisers Hyderabad,Yuvraj Singh,DA Warner,TS Mills,S Dhawan,Sunrisers Hyderabad,0,0,0,0
1,Hyderabad,2017-04-05,Indian Premier League,T20,Sunrisers Hyderabad,Yuvraj Singh,DA Warner,TS Mills,S Dhawan,Sunrisers Hyderabad,0,0,0,0
2,Hyderabad,2017-04-05,Indian Premier League,T20,Sunrisers Hyderabad,Yuvraj Singh,DA Warner,TS Mills,S Dhawan,Sunrisers Hyderabad,0,4,0,4
3,Hyderabad,2017-04-05,Indian Premier League,T20,Sunrisers Hyderabad,Yuvraj Singh,DA Warner,TS Mills,S Dhawan,Sunrisers Hyderabad,0,0,0,0
4,Hyderabad,2017-04-05,Indian Premier League,T20,Sunrisers Hyderabad,Yuvraj Singh,DA Warner,TS Mills,S Dhawan,Sunrisers Hyderabad,0,0,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260915,Bangalore,2016-05-29,Indian Premier League,T20,Sunrisers Hyderabad,BCJ Cutting,Sachin Baby,B Kumar,CJ Jordan,Royal Challengers Bangalore,19,2,0,2
260916,Bangalore,2016-05-29,Indian Premier League,T20,Sunrisers Hyderabad,BCJ Cutting,Sachin Baby,B Kumar,CJ Jordan,Royal Challengers Bangalore,19,0,0,0
260917,Bangalore,2016-05-29,Indian Premier League,T20,Sunrisers Hyderabad,BCJ Cutting,Iqbal Abdulla,B Kumar,Sachin Baby,Royal Challengers Bangalore,19,0,1,1
260918,Bangalore,2016-05-29,Indian Premier League,T20,Sunrisers Hyderabad,BCJ Cutting,Sachin Baby,B Kumar,Iqbal Abdulla,Royal Challengers Bangalore,19,1,0,1


In [7]:
json_folder = 'odis_json'
match_sum = match_summary(json_folder)
odi_data_table = match_sum.get_over_data()

In [8]:
odi_data_table

,city,date,event_name,match_type,winner,player_of_match,batter,bowler,non_striker,inning,over,runs_batter,runs_extras,runs_total
0,Brisbane,2017-01-13,Pakistan in Australia ODI Series,ODI,Australia,MS Wade,DA Warner,Mohammad Amir,TM Head,Australia,0,0,0,0
1,Brisbane,2017-01-13,Pakistan in Australia ODI Series,ODI,Australia,MS Wade,DA Warner,Mohammad Amir,TM Head,Australia,0,0,0,0
2,Brisbane,2017-01-13,Pakistan in Australia ODI Series,ODI,Australia,MS Wade,DA Warner,Mohammad Amir,TM Head,Australia,0,0,0,0
3,Brisbane,2017-01-13,Pakistan in Australia ODI Series,ODI,Australia,MS Wade,DA Warner,Mohammad Amir,TM Head,Australia,0,0,0,0
4,Brisbane,2017-01-13,Pakistan in Australia ODI Series,ODI,Australia,MS Wade,DA Warner,Mohammad Amir,TM Head,Australia,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1530106,Edinburgh,2016-08-16,ICC World Cricket League Championship,ODI,Scotland,Unknown,PL Mommsen,Mohammad Shahzad,RD Berrington,Scotland,46,0,0,0
1530107,Edinburgh,2016-08-16,ICC World Cricket League Championship,ODI,Scotland,Unknown,RD Berrington,Rohan Mustafa,PL Mommsen,Scotland,47,0,0,0
1530108,Edinburgh,2016-08-16,ICC World Cricket League Championship,ODI,Scotland,Unknown,RD Berrington,Rohan Mustafa,PL Mommsen,Scotland,47,0,0,0
1530109,Edinburgh,2016-08-16,ICC World Cricket League Championship,ODI,Scotland,Unknown,RD Berrington,Rohan Mustafa,PL Mommsen,Scotland,47,0,0,0


In [9]:
json_folder = 't20s_json'
match_sum = match_summary(json_folder)
t20_data_table = match_sum.get_over_data()

In [10]:
t20_data_table

,city,date,event_name,match_type,winner,player_of_match,batter,bowler,non_striker,inning,over,runs_batter,runs_extras,runs_total
0,Unknown,2017-02-17,Sri Lanka in Australia T20I Series,T20,Sri Lanka,DAS Gunaratne,AJ Finch,SL Malinga,M Klinger,Australia,0,0,0,0
1,Unknown,2017-02-17,Sri Lanka in Australia T20I Series,T20,Sri Lanka,DAS Gunaratne,AJ Finch,SL Malinga,M Klinger,Australia,0,0,0,0
2,Unknown,2017-02-17,Sri Lanka in Australia T20I Series,T20,Sri Lanka,DAS Gunaratne,AJ Finch,SL Malinga,M Klinger,Australia,0,1,0,1
3,Unknown,2017-02-17,Sri Lanka in Australia T20I Series,T20,Sri Lanka,DAS Gunaratne,M Klinger,SL Malinga,AJ Finch,Australia,0,2,0,2
4,Unknown,2017-02-17,Sri Lanka in Australia T20I Series,T20,Sri Lanka,DAS Gunaratne,M Klinger,SL Malinga,AJ Finch,Australia,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
903782,Colombo,2016-09-09,Australia in Sri Lanka T20I Series,T20,Australia,GJ Maxwell,TM Head,SS Pathirana,PM Nevill,Australia,17,1,0,1
903783,Colombo,2016-09-09,Australia in Sri Lanka T20I Series,T20,Australia,GJ Maxwell,PM Nevill,SS Pathirana,TM Head,Australia,17,3,0,3
903784,Colombo,2016-09-09,Australia in Sri Lanka T20I Series,T20,Australia,GJ Maxwell,TM Head,SS Pathirana,PM Nevill,Australia,17,0,0,0
903785,Colombo,2016-09-09,Australia in Sri Lanka T20I Series,T20,Australia,GJ Maxwell,TM Head,SS Pathirana,PM Nevill,Australia,17,0,0,0


In [11]:
json_folder = 'tests_json'
match_sum = match_summary(json_folder)
test_data_table = match_sum.get_over_data()

In [12]:
test_data_table

,city,date,event_name,match_type,winner,player_of_match,batter,bowler,non_striker,inning,over,runs_batter,runs_extras,runs_total
0,Perth,2016-11-03,South Africa in Australia Test Series,Test,South Africa,K Rabada,SC Cook,MA Starc,D Elgar,South Africa,0,0,0,0
1,Perth,2016-11-03,South Africa in Australia Test Series,Test,South Africa,K Rabada,SC Cook,MA Starc,D Elgar,South Africa,0,0,0,0
2,Perth,2016-11-03,South Africa in Australia Test Series,Test,South Africa,K Rabada,SC Cook,MA Starc,D Elgar,South Africa,0,0,0,0
3,Perth,2016-11-03,South Africa in Australia Test Series,Test,South Africa,K Rabada,SC Cook,MA Starc,D Elgar,South Africa,0,0,0,0
4,Perth,2016-11-03,South Africa in Australia Test Series,Test,South Africa,K Rabada,HM Amla,MA Starc,D Elgar,South Africa,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1667314,Colombo,2016-08-13,Warne-Muralitharan Trophy,Test,Sri Lanka,HMRKB Herath,JM Holland,MDK Perera,NM Lyon,Australia,43,0,0,0
1667315,Colombo,2016-08-13,Warne-Muralitharan Trophy,Test,Sri Lanka,HMRKB Herath,JM Holland,MDK Perera,NM Lyon,Australia,43,0,0,0
1667316,Colombo,2016-08-13,Warne-Muralitharan Trophy,Test,Sri Lanka,HMRKB Herath,JM Holland,MDK Perera,NM Lyon,Australia,43,0,0,0
1667317,Colombo,2016-08-13,Warne-Muralitharan Trophy,Test,Sri Lanka,HMRKB Herath,JM Holland,MDK Perera,NM Lyon,Australia,43,0,0,0
